# CARD Demo Notebook

Explore a random sample: neighbor poses (±1), stereo pairing, GT depth, ground plane, and convert depth to height above ground.

In [ ]:
import os, random, numpy as np
import torch
import cv2
from pathlib import Path
from card_dataset.dataloader import UnifiedCardDataset


def chw_to_hwc_uint8(img_t, mean, std):
    img = img_t.detach().cpu().float().numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img * 255.0, 0, 255).astype(np.uint8)
    return img


def compute_height_from_ground_plane(depth_pred, depth_gt, K, T_cam_world, ground_plane_info):
    """Compute signed height above the supplied ground plane for valid depth pixels."""
    plane_normal_world = ground_plane_info["plane_normal_world"]
    if isinstance(plane_normal_world, torch.Tensor):
        plane_normal_world = plane_normal_world.detach().cpu().numpy()
    else:
        plane_normal_world = np.asarray(plane_normal_world)

    plane_point_world = ground_plane_info["plane_point_world"]
    if isinstance(plane_point_world, torch.Tensor):
        plane_point_world = plane_point_world.detach().cpu().numpy()
    else:
        plane_point_world = np.asarray(plane_point_world)

    normal_norm = np.linalg.norm(plane_normal_world)
    if not np.isfinite(normal_norm) or normal_norm <= 1e-8:
        raise ValueError("Ground-plane normal must be finite and nonzero")
    plane_normal_unit = plane_normal_world / normal_norm

    def depth_to_height(depth_map):
        if depth_map is None:
            return None
        depth_map = np.asarray(depth_map)
        h, w = depth_map.shape
        valid_mask = (depth_map > 0) & np.isfinite(depth_map)
        height_map = np.zeros_like(depth_map, dtype=np.float32)
        if not np.any(valid_mask):
            return height_map
        u, v = np.meshgrid(np.arange(w), np.arange(h))
        valid_u = u[valid_mask]
        valid_v = v[valid_mask]
        valid_depth = depth_map[valid_mask]
        x_cam = (valid_u - K[0, 2]) * valid_depth / K[0, 0]
        y_cam = (valid_v - K[1, 2]) * valid_depth / K[1, 1]
        points_cam_hom = np.stack([x_cam, y_cam, valid_depth, np.ones_like(valid_depth)], axis=-1)
        points_world = (T_cam_world @ points_cam_hom.T).T[:, :3]
        vectors_to_points = points_world - plane_point_world.reshape(1, 3)
        height_map[valid_mask] = vectors_to_points @ plane_normal_unit
        return height_map

    return depth_to_height(depth_pred), depth_to_height(depth_gt)

## Configure Paths

The args to the data loader, the data root, and the split file to be read

In [ ]:
HEIGHT, WIDTH = 1000, 1216
LOAD_NEIGHBORS = True
LOAD_GROUND_PLANE = True
USE_STEREO = True

In [ ]:
import os
DATA_ROOT = str(Path(os.environ.get('CARD_DATA_ROOT', './data')).expanduser().resolve())
SPLIT_FILE = os.environ.get('CARD_SPLIT_FILE', str(Path('splits/val.txt').resolve()))
print('DATA_ROOT =', DATA_ROOT)
print('SPLIT_FILE =', SPLIT_FILE)

In [ ]:
# Resolve split path
split_path = Path(SPLIT_FILE)
if not split_path.is_absolute():
    split_path = Path(DATA_ROOT) / split_path

ds = UnifiedCardDataset.for_visualization(
    base_root=DATA_ROOT,
    split_file=str(split_path.relative_to(DATA_ROOT)) if str(split_path).startswith(str(Path(DATA_ROOT))) else str(split_path),
    height=HEIGHT, width=WIDTH,
    load_neighbors=LOAD_NEIGHBORS,
    load_ground_plane_info=LOAD_GROUND_PLANE,
    stereo=USE_STEREO,
    context_offsets=(-1, 0, +1) if LOAD_NEIGHBORS else (0,),
)
print('Dataset length:', len(ds))

## What you can get from the data loader?

timestamp, the pose for that image, and by choice you can also load the neighbours (offset +1/-1) and stereo images as well. 

In [ ]:
idx = random.randrange(len(ds)) #replace this by specific index if that's needed
item = ds[idx]
center_cam = item['meta']['center_cam']
frames = item[center_cam]['frames']
f0 = frames[0]
mean = ds.rgb_mean[0, 0, :]
std = ds.rgb_std[0, 0, :]
rgb0 = chw_to_hwc_uint8(f0['rgb'], mean, std)
print('Center timestamp:', f0['timestamp_us'])
print('Center K shape:', tuple(f0['K'].shape))
print('Center T_cam_world shape:', tuple(f0['T_cam_world'].shape))
rgb0.shape

In [ ]:
rels = item[center_cam].get('rel_poses_to_center', {})
print('Neighbor offsets:', list(frames.keys()))
for off, rel in rels.items():
    print(f'Offset {off:+d} relative pose to center: shape', tuple(rel.shape))

for off in (-1, +1):
    f = frames.get(off)
    if f is not None:
        print(f'Neighbor {off:+d} timestamp:', f['timestamp_us'])
    else:
        print(f'Neighbor {off:+d} not available')

In [ ]:
st = item.get('stereo', None)
if st is not None:
    print('Stereo baseline (m):', st['baseline_m'])
    print('Left:', st['left_cam'], 'Right:', st['right_cam'])
else:
    print('Stereo not available for this sample')

## Depth & Height visualization

What you can load is actually depth, (agg depth is the quasi-dense depth), while you can still get the single depth as well if needed. 
You can also conver to height by the ground plane information, this is based on the current 4 wheel positions, it may not be the actual ground in front of the vehicle, but it is calculated from the plane that is touched by the 4 wheels at this moment, so that's why it may be different due to the continous suspension. 

In [ ]:
def pick_depth(frame):
    # Prefer aggregated, then single, then lidar sparse
    for key in ['depth_agg', 'depth_single', 'lidar_sparse']:
        t = frame.get(key, None)
        if t is not None:
            d = t[0].detach().cpu().numpy()
            return key, d
    return None, None


dkey, depth_gt = pick_depth(f0)
print('Selected depth key:', dkey)
if depth_gt is not None:
    print('Depth shape:', depth_gt.shape, 'min/max:', float(depth_gt.min()), float(depth_gt.max()))
else:
    print('No depth available in center frame')

In [ ]:
gp = f0.get('ground_plane_info', None)
if gp is not None and depth_gt is not None:
    K = f0['K'].detach().cpu().numpy()
    Tcw = f0['T_cam_world'].detach().cpu().numpy()
    height_pred, height_gt = compute_height_from_ground_plane(None, depth_gt, K, Tcw, gp)
    print('Height map stats (GT): min/max', float(np.min(height_gt)), float(np.max(height_gt)))
else:
    print('Ground plane info or depth_gt missing')


In [ ]:
# Visualization helpers
import numpy as np, cv2
from pathlib import Path


def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)


def save_png(path: Path, rgb: np.ndarray):
    ensure_dir(path.parent)
    cv2.imwrite(str(path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))


def depth_to_colormap_cv(depth: np.ndarray, vmin: float = 0.1, vmax: float = 60.0,) -> np.ndarray:
    d = np.nan_to_num(depth, nan=0.0)
    h, w = d.shape
    cm = np.zeros((h, w, 3), np.uint8)
    valid = d > 0
    if np.any(valid):
        dn = np.clip((d - vmin) / max(1e-6, (vmax - vmin)), 0.0, 1.0)
        d8 = (dn * 255.0).astype(np.uint8)
        cm_full = cv2.applyColorMap(d8, cv2.COLORMAP_TURBO)[:, :, ::-1]
        cm[valid] = cm_full[valid]
    cm[~valid] = np.array([255, 255, 255], np.uint8)
    return cm


def height_to_colormap(height: np.ndarray, vmin: float = -1.0, vmax: float = 3.0,
                        invalid_mask: np.ndarray | None = None,
                        zero_highlight_eps: float = 0.03) -> np.ndarray:
    h = np.nan_to_num(height, nan=0.0)
    H, W = h.shape
    cm = np.zeros((H, W, 3), np.uint8)
    hn = np.clip((h - vmin) / max(1e-6, (vmax - vmin)), 0.0, 1.0)
    h8 = (hn * 255.0).astype(np.uint8)
    cm_full = cv2.applyColorMap(h8, cv2.COLORMAP_TURBO)[:, :, ::-1]
    cm[:, :] = cm_full
    # highlight near-zero band as WHITE
    if zero_highlight_eps is not None and zero_highlight_eps > 0:
        band = np.abs(h) <= zero_highlight_eps
        cm[band] = np.array([255, 255, 255], np.uint8)
    if invalid_mask is not None:
        cm[invalid_mask] = np.array([255, 255, 255], np.uint8)
    return cm



def overlay(rgb: np.ndarray, cm: np.ndarray, alpha: float = 0.6) -> np.ndarray:
    return (alpha * cm.astype(np.float32) + (1 - alpha) * rgb.astype(np.float32)).astype(np.uint8)


def backproject_depth(depth: np.ndarray, K: np.ndarray):
    H, W = depth.shape
    u, v = np.meshgrid(np.arange(W), np.arange(H))
    m = depth > 0
    if not np.any(m):
        return np.zeros((0, 3), np.float32), np.zeros((0, 2), np.float32)
    z = depth[m].astype(np.float32)
    u = u[m].astype(np.float32); v = v[m].astype(np.float32)
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    x = (u - cx) * z / fx
    y = (v - cy) * z / fy
    return np.stack([x, y, z], 1), np.stack([u, v], 1)


def transform_points(pts: np.ndarray, T4x4: np.ndarray) -> np.ndarray:
    if pts.size == 0:
        return pts
    N = pts.shape[0]
    ph = np.hstack([pts, np.ones((N, 1), np.float32)])
    return (T4x4 @ ph.T).T[:, :3]


def project_points(pts_cam: np.ndarray, K: np.ndarray, hwc_shape: tuple[int, int, int]) -> tuple[np.ndarray, np.ndarray]:
    H, W = hwc_shape[0], hwc_shape[1]
    if pts_cam.size == 0:
        return np.zeros((0, 2), np.float32), np.zeros((0,), bool)
    z = pts_cam[:, 2]
    m = z > 1e-6
    z = np.maximum(z, 1e-6)
    u = (K[0, 0] * pts_cam[:, 0] / z) + K[0, 2]
    v = (K[1, 1] * pts_cam[:, 1] / z) + K[1, 2]
    inb = (u >= 0) & (u < W) & (v >= 0) & (v < H)
    m = m & inb
    return np.stack([u, v], 1), m


def draw_points(img: np.ndarray, uv: np.ndarray, mask: np.ndarray, color=(0, 255, 0), r: int = 1) -> np.ndarray:
    out = img.copy()
    if uv.shape[0] == 0:
        return out
    pts = uv[mask].astype(np.int32)
    for x, y in pts:
        cv2.circle(out, (int(x), int(y)), r, color, -1)
    return out


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

def show_img(title: str, img: np.ndarray, figsize=(8, 6)):
    plt.figure(figsize=figsize)
    plt.title(title)
    plt.axis('off')
    if img.ndim == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(img)
    plt.show()


def show_grid(pairs: list[tuple[str, np.ndarray]], ncols: int = 3, figsize=(16, 10), 
              vmin=None, vmax=None, cmap='turbo'):
    
    n = len(pairs)
    if n == 0: return

    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    
    for i, (title, img) in enumerate(pairs):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        ax.set_title(title)
        ax.axis('off')
        
        # If image is 2D (raw depth), use vmin/vmax to colorize it.
        # If image is 3D (RGB), vmin/vmax are ignored by imshow but valid for the colorbar.
        if img.ndim == 2:
            ax.imshow(img, cmap=cmap if vmin else 'gray', vmin=vmin, vmax=vmax)
        else:
            ax.imshow(img)

    # Hide empty subplots
    for j in range(i + 1, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r][c].axis('off')

    # Add Colorbar if range is provided
    if vmin is not None and vmax is not None:
        norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        cb = fig.colorbar(sm, ax=axes, shrink=0.8, aspect=30, label='(m)')

    # plt.tight_layout() # strict tight_layout sometimes fights with fig.colorbar
    plt.show()

In [ ]:

# Center RGB
center_img = rgb0.copy()

# Depth colormap + overlay
imgs = [("Center RGB", center_img)]
if depth_gt is not None:
    d_cm = depth_to_colormap_cv(depth_gt, vmin=0.1, vmax=60.0)
    #make the baackground color as white or invalid
    d_cm = np.where(d_cm == np.array([255, 255, 255], np.uint8), np.array([255, 255, 255], np.uint8), d_cm)
    d_ov = overlay(center_img, d_cm, 0.6)
    imgs += [("Depth Colormap", d_cm), ("Depth Overlay", d_ov)]
else:
    print('No depth available to visualize')
show_grid(imgs, ncols=3, figsize=(18, 10))
#show_grid(imgs, ncols=3, vmin=0.1, vmax=60.0, cmap='turbo') if you want to see the colorbar range.


## Projecting the depth we have for image at (t) and project it to (t+1) and (t-1) to test the calibiration as well as the 6 DOF pose we have. 

you can see we get the depth for one image and project it on the neighbours, and for any given image, it is visible that everything is consistent. 

In [ ]:
# Inline: Backproject center depth and reproject to center and neighbors
K0 = f0['K'].detach().cpu().numpy()
Tcw0 = f0['T_cam_world'].detach().cpu().numpy()
Twc0 = np.linalg.inv(Tcw0)
pts_c, _ = backproject_depth(depth_gt, K0)
Pw = transform_points(pts_c, Tcw0)  # camera@t -> world

# center reprojection
Pc0 = transform_points(Pw, Twc0)
uv0, m0 = project_points(Pc0, K0, center_img.shape)
img_center_pts = draw_points(center_img, uv0, m0, color=(0, 255, 0), r=1)

grid = [("Reproj Center Points", img_center_pts)]

# neighbors reprojection
for off, tag, color in [(-1, 'minus1', (255, 200, 0)), (1, 'plus1', (255, 0, 200))]:
    fn = frames.get(off)
    if fn is None or fn.get('rgb') is None:
        continue
    rgb_n = chw_to_hwc_uint8(fn['rgb'], ds.rgb_mean[0,0,:], ds.rgb_std[0,0,:])
    Kn = fn['K'].detach().cpu().numpy()
    Tcwn = fn['T_cam_world'].detach().cpu().numpy()
    Twcn = np.linalg.inv(Tcwn)
    Pcn = transform_points(Pw, Twcn)
    uvn, mn = project_points(Pcn, Kn, rgb_n.shape)
    img_n_pts = draw_points(rgb_n, uvn, mn, color=color, r=1)
    title = f"Reproj {tag} Points"
    grid.append((title, img_n_pts))

show_grid(grid, ncols=3, figsize=(18, 10))


In [ ]:
invalid = ~(np.isfinite(depth_gt) & (depth_gt > 0))
h_cm = height_to_colormap(height_gt, vmin=-1.0, vmax=3.0, invalid_mask=invalid)

h_ov = overlay(center_img, h_cm, 0.6)
show_grid([("Height Colormap", h_cm), ("Height Overlay", h_ov)], ncols=2, figsize=(14, 6))
#show_grid([("Height Colormap", h_cm), ("Height Overlay", h_ov)], ncols=2, figsize=(14, 6), vmin=-1.0, vmax=3.0, cmap='turbo')